# FatigueSet Pipeline Tutorial

Este notebook enseña cómo utilizar el `FatigueSetPipeline`, un orquestador de extremo a extremo que cubre todos los pasos del procesamiento del dataset FatigueSet:

1. **Carga** del dataset completo
2. **Validación** de integridad de datos
3. **Cálculo de fatigabilidad** (deltas entre fases)
4. **Construcción del dataset ML** agregado por participante, sesión y fase
5. **Normalización** de features
6. **Ventaneo** temporal con extracción de estadísticas
7. **Análisis de correlaciones** con fatiga física y mental

Usaremos datos sintéticos mínimos para demostrar cada paso sin dependencias externas.

## Importar Librerías Necesarias

In [1]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Agregar la librería al path si es necesario
try:
    from fatigueset import FatigueSetPipeline
except ImportError:
    # Si no está en el path, agregamos la carpeta de la librería
    lib_path = Path.cwd().parent.parent / "fatigueset-lib"
    sys.path.insert(0, str(lib_path))
    from fatigueset import FatigueSetPipeline

print("✓ Librerías importadas exitosamente")

✓ Librerías importadas exitosamente


## Parte 1: Crear un dataset sintético mínimo

Crearemos una estructura de datos pequeña pero completa para demostrar el pipeline sin necesidad del dataset real.

In [2]:
import tempfile
from pathlib import Path

# Crear directorio temporal con estructura FatigueSet
tmpdir = Path(tempfile.mkdtemp())
base = tmpdir / 'fatigueset'
participant_dir = base / '01' / '01'
participant_dir.mkdir(parents=True, exist_ok=True)

print(f"Directorio de prueba creado en: {base}")

# Helper para escribir CSV
def write_csv(path, content):
    path.write_text(content, encoding='utf-8')

# Metadata: mapeo de participante -> sesiones por intensidad
write_csv(
    base / 'metadata.csv',
    'participant_id,low_session,medium_session,high_session\n1,1,2,3\n'
)

# Fatiga subjetiva: 3 mediciones por sesión (M1, M2, M3)
write_csv(
    participant_dir / 'exp_fatigue.csv',
    '''measurementNumber,physicalFatigueScore,mentalFatigueScore
0,20,25
1,35,45
2,55,70
'''
)

# Sensores cardiacos (chest)
write_csv(
    participant_dir / 'chest_physiology_summary.csv',
    '''timestamp,hr,br,hrv
1,60,12,40
2,62,13,42
3,65,14,45
4,68,15,48
'''
)

# EDA (Electrodermal Activity - muñeca)
write_csv(
    participant_dir / 'wrist_eda.csv',
    '''eda
0.1
0.15
0.2
0.25
0.3
'''
)

# N-Back (tarea cognitiva)
write_csv(
    participant_dir / 'exp_nback.csv',
    '''measurementNumber,isCorrectResponse,responseTime
0,1,500
0,1,480
0,0,650
1,1,520
1,1,490
1,0,600
2,1,510
2,1,500
'''
)

# CRT (Choice Reaction Time)
write_csv(
    participant_dir / 'exp_crt.csv',
    '''measurementNumber,isCorrectResponse,responseTime
0,1,300
0,1,310
1,1,320
1,1,330
2,1,315
2,0,400
'''
)

print("✓ Dataset sintético creado con 6 archivos")

Directorio de prueba creado en: C:\Users\egull\AppData\Local\Temp\tmplpn7xytk\fatigueset
✓ Dataset sintético creado con 6 archivos


## Parte 2: Inicializar el Pipeline

El `FatigueSetPipeline` es un orquestador que gestiona todas las etapas del procesamiento. Lo inicializamos especificando:
- **dataset_path**: ruta al dataset (en nuestro caso, el temporal)
- **participantes**: lista de IDs de participantes a procesar
- **sesiones**: lista de sesiones
- **umbral_nulos**: porcentaje máximo de valores nulos permitido para validar

In [3]:
# Inicializar el pipeline
pipeline = FatigueSetPipeline(
    dataset_path=str(base),
    participantes=['01'],      # Solo participante 01
    sesiones=['01'],           # Solo sesión 01
    umbral_nulos=5.0           # Máx 5% de valores nulos
)

print("✓ Pipeline inicializado")
print(f"  - Dataset path: {base}")
print(f"  - Participantes: {pipeline.participantes}")
print(f"  - Sesiones: {pipeline.sesiones}")

✓ Pipeline inicializado
  - Dataset path: C:\Users\egull\AppData\Local\Temp\tmplpn7xytk\fatigueset
  - Participantes: ['01']
  - Sesiones: ['01']


## Parte 3: Ejecutar el Pipeline Completo

El método `ejecutar()` cubre todos los pasos de un extremo a otro y devuelve un diccionario con todos los artefactos generados.

In [4]:
resultados = pipeline.ejecutar(
    verbose=False,
    incluir_ventanas=True,     # Activar ventaneo temporal
    window_size=2,             # Ventanas de 2 filas
    step=1,                    # Solapamiento de 50%
    normalizar=True,           # Normalizar features
    metodo_normalizacion='zscore'  # Z-score normalization
)

print("✓ Pipeline ejecutado exitosamente")
print("\nArtefactos generados:")
for key in resultados.keys():
    if isinstance(resultados[key], dict):
        print(f"  - {key}: {len(resultados[key])} elementos")
    elif isinstance(resultados[key], pd.DataFrame):
        print(f"  - {key}: {resultados[key].shape[0]} filas × {resultados[key].shape[1]} columnas")
    else:
        print(f"  - {key}: {type(resultados[key]).__name__}")

✓ Pipeline ejecutado exitosamente

Artefactos generados:
  - dataset: 10 elementos
  - validacion: 5 filas × 7 columnas
  - resumen_validacion: 5 filas × 4 columnas
  - problemas_validacion: 0 filas × 7 columnas
  - fatigabilidad: 1 filas × 16 columnas
  - ml: 3 filas × 32 columnas
  - ml_normalizado: 3 filas × 32 columnas
  - ventanas: 0 filas × 0 columnas
  - correlaciones: 2 elementos


## Parte 4: Explorar el Dataset Cargado

Veamos qué datos se han cargado de cada sensor y tarea cognitiva.

In [5]:
dataset = resultados['dataset']

print("Contenido del dataset cargado:\n")
if dataset['fatiga'] is not None:
    print("exp_fatigue:")
    print(dataset['fatiga'][['measurementNumber', 'physicalFatigueScore', 'mentalFatigueScore']])
    print()

if dataset['chest'] is not None:
    print("chest_physiology_summary:")
    print(dataset['chest'][['hr', 'br', 'hrv']].head(3))
    print()

if dataset['wrist'] and dataset['wrist']['eda'] is not None:
    print("wrist_eda:")
    print(dataset['wrist']['eda'][['eda']].head(3))
    print()

if dataset['nback'] is not None:
    print("exp_nback:")
    print(dataset['nback'][['measurementNumber', 'isCorrectResponse', 'responseTime']].head(3))
    print()

Contenido del dataset cargado:

exp_fatigue:
   measurementNumber  physicalFatigueScore  mentalFatigueScore
0                  0                    20                  25
1                  1                    35                  45
2                  2                    55                  70

chest_physiology_summary:
   hr  br  hrv
0  60  12   40
1  62  13   42
2  65  14   45

wrist_eda:
    eda
0  0.10
1  0.15
2  0.20

exp_nback:
   measurementNumber  isCorrectResponse  responseTime
0                  0                  1           500
1                  0                  1           480
2                  0                  0           650



## Parte 5: Validación de Integridad de Datos

El pipeline valida automáticamente:
- Presencia de archivos críticos
- Porcentaje de valores nulos
- Duplicados en los datos

In [6]:
df_validacion = resultados['validacion']
resumen_validacion = resultados['resumen_validacion']

print("Resumen de Validación:")
print(resumen_validacion)
print(f"\nTotal de validaciones: {len(df_validacion)}")
print(f"Exitosas: {(df_validacion['Estado'] == '✓').sum()}")
print(f"Con alertas: {(df_validacion['Estado'] == '⚠️').sum()}")

problemas = resultados['problemas_validacion']
if not problemas.empty:
    print("\nArchivos con problemas:")
    print(problemas)

Resumen de Validación:
                  Sesiones_OK  Sesiones_total  Nulos_pct_max  Duplicados_max
Archivo                                                                     
CRT Task                    1               1            0.0               0
Chest Physiology            1               1            0.0               0
Fatigue Scores              1               1            0.0               0
N-Back Task                 1               1            0.0               0
Wrist EDA                   1               1            0.0               0

Total de validaciones: 5
Exitosas: 5
Con alertas: 0


## Parte 6: Fatigabilidad - Deltas Entre Fases

La fatigabilidad se calcula como los cambios (deltas) en los scores de fatiga entre las 3 fases de medición:
- **M1**: Baseline
- **M2**: Post-ejercicio
- **M3**: Post-tarea cognitiva

In [7]:
df_fatigabilidad = resultados['fatigabilidad']

if not df_fatigabilidad.empty:
    print("Fatigabilidad calculada (deltas de fatiga entre fases):\n")
    cols_delta = [col for col in df_fatigabilidad.columns if 'delta' in col]
    print(df_fatigabilidad[['participante', 'sesion', 'intensidad'] + cols_delta])
    
    print("\n📊 Interpretación:")
    print("- delta_fisica_ejercicio: cambio físico después del ejercicio (M2-M1)")
    print("- delta_mental_ejercicio: cambio mental después del ejercicio (M2-M1)")
    print("- delta_fisica_cognitivo: cambio físico tras tarea cognitiva (M3-M2)")
    print("- delta_mental_cognitivo: cambio mental tras tarea cognitiva (M3-M2)")
    print("- delta_fisica_total: cambio físico acumulado (M3-M1)")
    print("- delta_mental_total: cambio mental acumulado (M3-M1)")
else:
    print("No hay datos de fatigabilidad disponibles")

Fatigabilidad calculada (deltas de fatiga entre fases):

  participante sesion intensidad  delta_fisica_ejercicio  \
0           01     01        low                      15   

   delta_mental_ejercicio  delta_fisica_cognitivo  delta_mental_cognitivo  \
0                      20                      20                      25   

   delta_fisica_total  delta_mental_total  
0                  35                  45  

📊 Interpretación:
- delta_fisica_ejercicio: cambio físico después del ejercicio (M2-M1)
- delta_mental_ejercicio: cambio mental después del ejercicio (M2-M1)
- delta_fisica_cognitivo: cambio físico tras tarea cognitiva (M3-M2)
- delta_mental_cognitivo: cambio mental tras tarea cognitiva (M3-M2)
- delta_fisica_total: cambio físico acumulado (M3-M1)
- delta_mental_total: cambio mental acumulado (M3-M1)


## Parte 7: Dataset ML Agregado

El dataset ML es la matriz final con:
- Una fila por (participante, sesión, fase) = 12 × 3 × 3 = 108 filas (con data completa)
- Features agregadas de todos los sensores
- Targets (fatiga física y mental)

In [8]:
df_ml = resultados['ml']

print(f"Dataset ML: {df_ml.shape[0]} filas × {df_ml.shape[1]} columnas\n")
print("Primeras 3 filas:")
print(df_ml.head(3))

print("\n📊 Columnas disponibles:")
cols_identidad = ['participante', 'sesion', 'intensidad', 'intensidad_num', 'fase', 'fase_num']
cols_targets = ['fatiga_fisica', 'fatiga_mental']
cols_features = [c for c in df_ml.columns if c not in cols_identidad + cols_targets]

print(f"- Identidad: {cols_identidad}")
print(f"- Targets: {cols_targets}")
print(f"- Features: {len(cols_features)} columnas")
if cols_features:
    print(f"  Primeras 5: {cols_features[:5]}")

Dataset ML: 3 filas × 32 columnas

Primeras 3 filas:
  participante sesion intensidad  intensidad_num                   fase  \
0           01     01        low               1            M1_baseline   
1           01     01        low               1      M2_post_ejercicio   
2           01     01        low               1  M3_post_fatiga_mental   

   fase_num  fatiga_fisica  fatiga_mental  hr_media   hr_std  ...  \
0         0             20             25      60.0      NaN  ...   
1         1             35             45      62.0      NaN  ...   
2         2             55             70      66.5  2.12132  ...   

   delta_fisica_cognitivo  delta_mental_cognitivo  delta_fisica_total  \
0                      20                      25                  35   
1                      20                      25                  35   
2                      20                      25                  35   

   delta_mental_total  fisica_M1  fisica_M2  fisica_M3  mental_M1  mental_M2

## Parte 8: Normalización de Features

Después de construir el dataset ML, el pipeline normaliza las features numéricas manteniendo los identificadores intactos.

In [9]:
df_ml_normalizado = resultados['ml_normalizado']

if not df_ml_normalizado.empty:
    print("Dataset ML Normalizado:")
    print(f"- Shape: {df_ml_normalizado.shape}")
    print(f"- Método: Z-score (media=0, std=1)")
    
    # Mostrar estadísticas de normalización
    cols_numericas = df_ml_normalizado.select_dtypes(include=[np.number]).columns
    stats = df_ml_normalizado[cols_numericas].describe()
    
    print("\nEstadísticas post-normalización (primeras 5 features):")
    print(stats[cols_numericas[:5]])
else:
    print("No hay dataset normalizado (puede estar vacío)")

Dataset ML Normalizado:
- Shape: (3, 32)
- Método: Z-score (media=0, std=1)

Estadísticas post-normalización (primeras 5 features):
       intensidad_num  fase_num  fatiga_fisica  fatiga_mental      hr_media
count             3.0       3.0       3.000000       3.000000  3.000000e+00
mean              1.0       1.0      36.666667      46.666667 -8.141636e-16
std               0.0       1.0      17.559423      22.546249  1.224745e+00
min               1.0       0.0      20.000000      25.000000 -1.042337e+00
25%               1.0       0.5      27.500000      35.000000 -6.744533e-01
50%               1.0       1.0      35.000000      45.000000 -3.065697e-01
75%               1.0       1.5      45.000000      57.500000  5.211684e-01
max               1.0       2.0      55.000000      70.000000  1.348907e+00


## Parte 9: Ventaneo Temporal

El ventaneo convierte series temporales en características estáticas mediante ventanas deslizantes:

In [10]:
df_ventanas = resultados['ventanas']

if not df_ventanas.empty:
    print(f"Ventanas creadas: {df_ventanas.shape[0]} ventanas × {df_ventanas.shape[1]} características\n")
    print("Primeras 3 ventanas:")
    print(df_ventanas.head(3))
    
    print("\n📊 Información sobre ventanas:")
    print(f"- Tamaño de ventana: {df_ventanas['ventana_tamano'].iloc[0] if len(df_ventanas) > 0 else 'N/A'}")
    print(f"- Rango de inicio: {df_ventanas['ventana_inicio'].min()} a {df_ventanas['ventana_inicio'].max()}")
    
    # Mostrar features de ventana
    feat_cols = [c for c in df_ventanas.columns if c not in ['ventana_inicio', 'ventana_fin', 'ventana_tamano', 'participante', 'sesion', 'fase']]
    print(f"- Características extraídas: {len(feat_cols)}")
    if feat_cols:
        print(f"  Ejemplos: {feat_cols[:5]}")
else:
    print("No hay ventanas (dataset ML vacío o muy pequeño para ventaneo)")

No hay ventanas (dataset ML vacío o muy pequeño para ventaneo)


## Parte 10: Análisis de Correlaciones

Finalmente, el pipeline identifica las features más correlacionadas con los targets (fatiga física y mental):

In [11]:
correlaciones = resultados['correlaciones']

print("Top 10 Features Correlacionadas con Targets:\n")

for target, top_features in correlaciones.items():
    if not top_features.empty:
        print(f"📊 {target.upper()}:")
        print(top_features.to_string())
        print()
    else:
        print(f"  {target}: sin correlaciones disponibles")
        print()

Top 10 Features Correlacionadas con Targets:

📊 FATIGA_MENTAL:
eda_std          -1.000
eda_media         1.000
br_media          0.999
fase_num          0.998
crt_rt            0.997
hr_media          0.988
hrv_media         0.988
nback_rt         -0.956
nback_accuracy    0.896
crt_accuracy     -0.896

📊 FATIGA_FISICA:
eda_std         -1.000
eda_media        1.000
fatiga_mental    1.000
br_media         0.999
crt_rt           0.999
fase_num         0.997
hr_media         0.991
hrv_media        0.991
nback_rt        -0.962
crt_accuracy    -0.904



## Parte 11: Ejecutar Pasos Individuales del Pipeline

También es posible ejecutar cada paso por separado si necesitas control fino. Aquí mostramos cómo acceder a cada método:

In [12]:
# Ejemplo: Ejecutar pasos individuales
print("Métodos disponibles en FatigueSetPipeline:\n")

metodos = [
    ('cargar_dataset', 'Carga todos los datos del dataset'),
    ('validar_dataset', 'Valida integridad de datos'),
    ('calcular_fatigabilidad', 'Calcula deltas de fatiga'),
    ('construir_dataset_ml', 'Construye dataset agregado'),
    ('normalizar_dataframe', 'Normaliza features'),
    ('crear_ventanas', 'Ventaneo temporal'),
    ('resumen_correlaciones', 'Análisis de correlaciones'),
    ('ejecutar', 'Ejecuta todo el pipeline'),
]

for metodo, descripcion in metodos:
    print(f"✓ {metodo:30} → {descripcion}")

print("\nEjemplo de uso individual:")
print("""
# Cargar dataset
dataset = pipeline.cargar_dataset(verbose=True)

# Validar
validacion = pipeline.validar_dataset()

# Calcular fatigabilidad
df_fatiga_subjetiva = dataset['fatiga']
df_fatigabilidad = pipeline.calcular_fatigabilidad(df_fatiga_subjetiva)

# Normalizar con min-max
df_ml_normalized = pipeline.normalizar_dataframe(
    df_ml, 
    metodo='minmax',
    group_cols=['participante', 'sesion']
)
""")

Métodos disponibles en FatigueSetPipeline:

✓ cargar_dataset                 → Carga todos los datos del dataset
✓ validar_dataset                → Valida integridad de datos
✓ calcular_fatigabilidad         → Calcula deltas de fatiga
✓ construir_dataset_ml           → Construye dataset agregado
✓ normalizar_dataframe           → Normaliza features
✓ crear_ventanas                 → Ventaneo temporal
✓ resumen_correlaciones          → Análisis de correlaciones
✓ ejecutar                       → Ejecuta todo el pipeline

Ejemplo de uso individual:

# Cargar dataset
dataset = pipeline.cargar_dataset(verbose=True)

# Validar
validacion = pipeline.validar_dataset()

# Calcular fatigabilidad
df_fatiga_subjetiva = dataset['fatiga']
df_fatigabilidad = pipeline.calcular_fatigabilidad(df_fatiga_subjetiva)

# Normalizar con min-max
df_ml_normalized = pipeline.normalizar_dataframe(
    df_ml, 
    metodo='minmax',
    group_cols=['participante', 'sesion']
)



## Resumen

El `FatigueSetPipeline` es un orquestador completo que:

1. **Carga** datos de múltiples sensores y tareas (Chest, Wrist, EEG, Ear, N-Back, CRT, etc.)
2. **Valida** integridad con reportes de nulos y duplicados
3. **Calcula fatigabilidad** como deltas entre fases M1→M2→M3
4. **Construye un dataset ML** agregado (108 filas en dataset completo)
5. **Normaliza** features por grupos de participante/sesión
6. **Ventanea temporalmente** series para extracción de estadísticas
7. **Analiza correlaciones** identificando features predictivas

El pipeline es **modular**: puedes ejecutarlo completo o paso a paso según tus necesidades. Todos los pasos están documentados y cuentan con manejo de errores automático.